# Huấn Luyện Benchmark Mô Hình UC-TTT & TTT Baseline Trên Google Colab
## (User-Credibility Temporal Tree Transformer vs Wu et al. PLOS ONE 2025)

Notebook này được thiết kế để chạy thực nghiệm phát hiện tin đồn trên mạng xã hội PHEME trên GPU đám mây của **Google Colab** (Tesla T4 / A100 / V100).

### 🌟 Các cải tiến chuẩn bài báo được tích hợp:
1. **Text Embedding**: `bert-base-uncased` (768 chiều, Max-Pooling theo công thức (1) của bài báo).
2. **Cửa sổ thời gian**: Chia động theo số bài viết ($k=8$ posts/cửa sổ), BiGRU hỗ trợ packed sequences cho độ dài biến thiên.
3. **Kiểm định LOEO**: 5 sự kiện đối chứng chuẩn Wu et al. 2025 (*charliehebdo, ferguson, germanwings-crash, ottawashooting, sydney-siege*).

---

### Bước 1: Kiểm tra phần cứng GPU được cấp phát

In [ ]:
!nvidia-smi

### Bước 2: Kết nối Google Drive để đồng bộ dữ liệu và lưu checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Bước 3: Clone Codebase từ GitHub (Nhánh `TTT-UC-TTT-v2-`)

In [ ]:
%cd /content
!rm -rf repo
!git clone -b TTT-UC-TTT-v2- https://github.com/tphu2029/SNA_EnhancingTTT-via-User-Credibility-Integration-for-Social-Media-Rumor-Detection.git repo
%cd /content/repo
!git status

### Bước 4: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q "transformers>=4.30.0,<4.49.0" scikit-learn pandas networkx tqdm

### Bước 5: Nạp dữ liệu từ Google Drive

Tạo một thư mục trên Google Drive của bạn tên là `PHEME_DATA` và tải lên:
- `pheme_processed.pkl` (~181 MB)
- *(Tùy chọn)* `text_embeddings_bert.pt` (~353 MB) nếu bạn đã có sẵn. Nếu chưa tải lên file embedding, ô Bước 5.2 bên dưới sẽ tự động trích xuất trực tiếp trên GPU Colab trong ~2 phút!

In [ ]:
import os
import shutil

DRIVE_DATA_DIR = "/content/drive/MyDrive/PHEME_DATA"
os.makedirs("data/processed", exist_ok=True)

# Copy dữ liệu cây PHEME
pkl_src = os.path.join(DRIVE_DATA_DIR, "pheme_processed.pkl")
pkl_dst = "data/processed/pheme_processed.pkl"
if os.path.exists(pkl_src):
    print("Đang copy pheme_processed.pkl...")
    shutil.copyfile(pkl_src, pkl_dst)
    print(f"-> Đã nạp: {pkl_dst} ({os.path.getsize(pkl_dst) / 1024 / 1024:.1f} MB)")
else:
    print(f"[!] Chưa tìm thấy {pkl_src}. Hãy tải file pheme_processed.pkl lên thư mục PHEME_DATA trên Google Drive!")

# Copy text embeddings nếu có
bert_src = os.path.join(DRIVE_DATA_DIR, "text_embeddings_bert.pt")
bert_dst = "data/processed/text_embeddings_bert.pt"
if os.path.exists(bert_src):
    print("Đang copy text_embeddings_bert.pt...")
    shutil.copyfile(bert_src, bert_dst)
    print(f"-> Đã nạp: {bert_dst} ({os.path.getsize(bert_dst) / 1024 / 1024:.1f} MB)")
else:
    print("[Thông báo] Chưa có sẵn text_embeddings_bert.pt trên Drive. Sẽ trích xuất tự động ở bước 5.2.")

### Bước 5.2 (Tùy chọn): Trích xuất BERT-base Embedding trên GPU Colab
*Chỉ chạy nếu chưa copy file `text_embeddings_bert.pt` từ Google Drive. Thời gian chạy ~2 phút trên GPU Tesla T4.*

In [ ]:
if not os.path.exists("data/processed/text_embeddings_bert.pt"):
    print("Đang trích xuất 768-d BERT embeddings với Max-Pooling...")
    !python experiments/extract_text_embeddings.py \
        --model_name bert-base-uncased \
        --batch_size 128 \
        --pooling max \
        --output data/processed/text_embeddings_bert.pt
    
    # Lưu ngược về Google Drive để lần sau không cần trích xuất lại
    if os.path.exists(DRIVE_DATA_DIR):
        print("Lưu bản sao embedding lên Google Drive...")
        shutil.copyfile("data/processed/text_embeddings_bert.pt", os.path.join(DRIVE_DATA_DIR, "text_embeddings_bert.pt"))
        print("Đã lưu thành công lên Drive!")
else:
    print("text_embeddings_bert.pt đã sẵn sàng!")

### Bước 6: Chạy Thực Nghiệm

Bạn có thể chọn chạy từng mô hình riêng biệt hoặc chạy toàn bộ bộ thử nghiệm đối chứng tự động.

#### Lựa chọn A: Chạy Tự Động Toàn Bộ Benchmark Đối Chứng (TTT Baseline vs UC-TTT Proposed)

In [ ]:
!python -u experiments/run_paper_sync_benchmarks.py \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --window_mode fixed_posts \
    --window_size 8 \
    --fusion_type residual_gate \
    --epochs 35 \
    --patience 8

#### Lựa chọn B: Hoặc Chạy Riêng Lẻ Từng Mô Hình

In [ ]:
# Chạy riêng TTT Baseline (Wu et al. 2025)
!python experiments/train_ttt_baseline.py \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --events_mode 5 \
    --window_mode fixed_posts \
    --window_size 8 \
    --d_model 768 \
    --nhead 12 \
    --num_tree_layers 6 \
    --epochs 35 \
    --patience 8

In [ ]:
# Chạy riêng UC-TTT Proposed (với Residual-Gate Fusion)
!python experiments/train_ucttt.py \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --events_mode 5 \
    --window_mode fixed_posts \
    --window_size 8 \
    --fusion_type residual_gate \
    --d_model 768 \
    --nhead 12 \
    --num_tree_layers 6 \
    --epochs 35 \
    --patience 8

### Bước 7: Sao Lưu Kết Quả & Checkpoints Về Google Drive

In [ ]:
BACKUP_DIR = "/content/drive/MyDrive/PHEME_DATA/results_backup"
os.makedirs(BACKUP_DIR, exist_ok=True)

!cp -r results/* "$BACKUP_DIR/"
print(f"Đã sao lưu toàn bộ kết quả và bảng so sánh vào: {BACKUP_DIR}")